In [13]:
import uproot
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import mplhep as hep
import gc 

In [14]:
# Leer archivos de datos reales de electrones (Run2012B y Run2012C)

file_mu_b = uproot.open("C:/Users/HP/Documents/Practica Profesional/Analisis BosonHiggs/Data Sets/Run2012B_DoubleMuParked.root")
tree_mu_b = file_mu_b['Events']
file_mu_c = uproot.open("C:/Users/HP/Documents/Practica Profesional/Analisis BosonHiggs/Data Sets/Run2012C_DoubleMuParked.root")
tree_mu_c = file_mu_c['Events']

file_el_b = uproot.open("C:/Users/HP/Documents/Practica Profesional/Analisis BosonHiggs/Data Sets/Run2012B_DoubleElectron.root")
tree_el_b = file_el_b['Events']
file_el_c = uproot.open("C:/Users/HP/Documents/Practica Profesional/Analisis BosonHiggs/Data Sets/Run2012C_DoubleElectron.root")
tree_el_c = file_el_c['Events']

print(tree_mu_b.keys())
print(tree_mu_c.keys())
print(tree_el_b.keys())
print(tree_el_c.keys())

['run', 'luminosityBlock', 'event', 'PV_npvs', 'PV_x', 'PV_y', 'PV_z', 'nMuon', 'Muon_pt', 'Muon_eta', 'Muon_phi', 'Muon_mass', 'Muon_charge', 'Muon_pfRelIso03_all', 'Muon_pfRelIso04_all', 'Muon_dxy', 'Muon_dxyErr', 'Muon_dz', 'Muon_dzErr', 'nElectron', 'Electron_pt', 'Electron_eta', 'Electron_phi', 'Electron_mass', 'Electron_charge', 'Electron_pfRelIso03_all', 'Electron_dxy', 'Electron_dxyErr', 'Electron_dz', 'Electron_dzErr', 'MET_pt', 'MET_phi']
['run', 'luminosityBlock', 'event', 'PV_npvs', 'PV_x', 'PV_y', 'PV_z', 'nMuon', 'Muon_pt', 'Muon_eta', 'Muon_phi', 'Muon_mass', 'Muon_charge', 'Muon_pfRelIso03_all', 'Muon_pfRelIso04_all', 'Muon_dxy', 'Muon_dxyErr', 'Muon_dz', 'Muon_dzErr', 'nElectron', 'Electron_pt', 'Electron_eta', 'Electron_phi', 'Electron_mass', 'Electron_charge', 'Electron_pfRelIso03_all', 'Electron_dxy', 'Electron_dxyErr', 'Electron_dz', 'Electron_dzErr', 'MET_pt', 'MET_phi']
['run', 'luminosityBlock', 'event', 'PV_npvs', 'PV_x', 'PV_y', 'PV_z', 'nMuon', 'Muon_pt', 'Mu

In [15]:
# Señal MC: H → ZZ* → 4μ y H → ZZ* → 4e
file_higgs = uproot.open("C:/Users/HP/Documents/Practica Profesional/Analisis BosonHiggs/Data Sets/SMHiggsToZZTo4L.root")
tree_higgs = file_higgs['Events']

# Fondo MC: ZZ → 2e2μ
file_zz_2e2mu = uproot.open("C:/Users/HP/Documents/Practica Profesional/Analisis BosonHiggs/Data Sets/ZZTo2e2mu.root")
tree_zz_2e2mu = file_zz_2e2mu['Events']

print(tree_higgs.keys())
print(tree_zz_2e2mu.keys())

['run', 'luminosityBlock', 'event', 'PV_npvs', 'PV_x', 'PV_y', 'PV_z', 'nMuon', 'Muon_pt', 'Muon_eta', 'Muon_phi', 'Muon_mass', 'Muon_charge', 'Muon_pfRelIso03_all', 'Muon_pfRelIso04_all', 'Muon_dxy', 'Muon_dxyErr', 'Muon_dz', 'Muon_dzErr', 'nElectron', 'Electron_pt', 'Electron_eta', 'Electron_phi', 'Electron_mass', 'Electron_charge', 'Electron_pfRelIso03_all', 'Electron_dxy', 'Electron_dxyErr', 'Electron_dz', 'Electron_dzErr', 'MET_pt', 'MET_phi']
['run', 'luminosityBlock', 'event', 'PV_npvs', 'PV_x', 'PV_y', 'PV_z', 'nMuon', 'Muon_pt', 'Muon_eta', 'Muon_phi', 'Muon_mass', 'Muon_charge', 'Muon_pfRelIso03_all', 'Muon_pfRelIso04_all', 'Muon_dxy', 'Muon_dxyErr', 'Muon_dz', 'Muon_dzErr', 'nElectron', 'Electron_pt', 'Electron_eta', 'Electron_phi', 'Electron_mass', 'Electron_charge', 'Electron_pfRelIso03_all', 'Electron_dxy', 'Electron_dxyErr', 'Electron_dz', 'Electron_dzErr', 'MET_pt', 'MET_phi']


In [7]:
# Extracción de variables físicas y de control para datos reales y MC

ramas_mu = ['run', 'luminosityBlock', 'event', 'nMuon', 'Muon_pt', 'Muon_eta', 'Muon_phi', 'Muon_charge', 'Muon_pfRelIso04_all', 'Muon_dxy', 'Muon_dz', 'Muon_dxyErr', 'Muon_dzErr']

# Cargar y filtrar Periodo B (Muones)

df_b_mu = tree_mu_b.arrays(ramas_mu, library="ak")

# Filtro: al menos 2 muones y eliminar aislamientos inválidos (-999 o negativos)
mask_b_mu = (ak.num(df_b_mu["Muon_pt"]) >= 2) & (ak.all(df_b_mu["Muon_pfRelIso04_all"] >= 0, axis=1))
df_b_mu_filtered = df_b_mu[mask_b_mu]

# Guardar en parquet de inmediato para proteger la RAM
ak.to_parquet(df_b_mu_filtered, "muones_periodo_b.parquet")

# Limpieza segura de RAM (evita el error si ya fue borrado antes)
try:
    del df_b_mu
except NameError:
    pass

try:
    del df_b_mu_filtered
except NameError:
    pass

gc.collect()

print("¡Periodo B procesado, guardado en Parquet y RAM liberada con éxito!")


¡Periodo B procesado, guardado en Parquet y RAM liberada con éxito!


In [8]:
# Cargar y filtrar Periodo C (Muones)
df_c_mu = tree_mu_c.arrays(ramas_mu, library="ak")

# Filtro: al menos 2 muones y eliminar aislamientos inválidos (-999 o negativos)
mask_c_mu = (ak.num(df_c_mu["Muon_pt"]) >= 2) & (ak.all(df_c_mu["Muon_pfRelIso04_all"] >= 0, axis=1))
df_c_mu_filtered = df_c_mu[mask_c_mu]

# Guardar en parquet de inmediato para proteger la RAM
ak.to_parquet(df_c_mu_filtered, "muones_periodo_c.parquet")

# Limpieza segura de RAM
try:
    del df_c_mu
except NameError:
    pass

try:
    del df_c_mu_filtered
except NameError:
    pass

gc.collect()

print("¡Periodo C procesado, guardado en Parquet y RAM liberada con éxito!")

¡Periodo C procesado, guardado en Parquet y RAM liberada con éxito!


In [4]:
# Cargar los archivos individuales ya filtrados desde el disco
df_b_mu_disk = ak.from_parquet("muones_periodo_b.parquet")
df_c_mu_disk = ak.from_parquet("muones_periodo_c.parquet")

In [5]:
# Concatenar ambos periodos en un solo Awkward Array
df_data_mu = ak.concatenate([df_b_mu_disk, df_c_mu_disk], axis=0)

In [6]:
# Guardar el archivo combinado final en disco
ak.to_parquet(df_data_mu, "data_muones_B_C_filtrados.parquet")
# Limpiar la RAM de los objetos temporales de lectura
del df_b_mu_disk, df_c_mu_disk
gc.collect()

192

In [7]:
# Extracción de variables físicas y de control para datos reales y MC

ramas_el = ['run', 'luminosityBlock', 'event', 'nElectron','Electron_pt', 'Electron_eta', 'Electron_phi', 'Electron_charge','Electron_pfRelIso03_all', 'Electron_dxy', 'Electron_dz', 'Electron_dxyErr', 'Electron_dzErr']

# Cargar y filtrar Periodo B (Electrones)

df_b_el = tree_el_b.arrays(ramas_el, library="ak")

# Filtro: al menos 2 muones y eliminar aislamientos inválidos (-999 o negativos)
mask_b_el = (ak.num(df_b_el["Electron_pt"]) >= 2) & (ak.all(df_b_el["Electron_pfRelIso03_all"] >= 0, axis=1))
df_b_el_filtered = df_b_el[mask_b_el]

# Guardar en parquet de inmediato para proteger la RAM
ak.to_parquet(df_b_el_filtered, "electrones_periodo_b.parquet")

# Limpieza segura de RAM (evita el error si ya fue borrado antes)
try:
    del df_b_el
except NameError:
    pass

try:
    del df_b_el_filtered
except NameError:
    pass

gc.collect()

print("¡Periodo B procesado, guardado en Parquet y RAM liberada con éxito!")


¡Periodo B procesado, guardado en Parquet y RAM liberada con éxito!


In [8]:
# Cargar y filtrar Periodo C (Electrones)
df_c_el = tree_el_c.arrays(ramas_el, library="ak")

# Filtro: al menos 2 muones y eliminar aislamientos inválidos (-999 o negativos)
mask_c_el = (ak.num(df_c_el["Electron_pt"]) >= 2) & (ak.all(df_c_el["Electron_pfRelIso03_all"] >= 0, axis=1))
df_c_el_filtered = df_c_el[mask_c_el]

# Guardar en parquet de inmediato para proteger la RAM
ak.to_parquet(df_c_el_filtered, "electrones_periodo_c.parquet")

# Limpieza segura de RAM
try:
    del df_c_el
except NameError:
    pass

try:
    del df_c_el_filtered
except NameError:
    pass

gc.collect()

print("¡Periodo C procesado, guardado en Parquet y RAM liberada con éxito!")

¡Periodo C procesado, guardado en Parquet y RAM liberada con éxito!


In [4]:
# Cargar los archivos individuales ya filtrados desde el disco
df_b_el_disk = ak.from_parquet("electrones_periodo_b.parquet")
df_c_el_disk = ak.from_parquet("electrones_periodo_c.parquet")

In [10]:
# Concatenar ambos periodos en un solo Awkward Array
df_data_el = ak.concatenate([df_b_el_disk, df_c_el_disk], axis=0)

In [11]:
# Guardar el archivo combinado final en disco
ak.to_parquet(df_data_el, "data_electrones_B_C_filtrados.parquet")
# Limpiar la RAM de los objetos temporales de lectura
del df_b_el_disk, df_c_el_disk
gc.collect()

3747

In [17]:
ramas_mu = ['run', 'luminosityBlock', 'event', 'nMuon','Muon_pt', 'Muon_eta', 'Muon_phi', 'Muon_charge', 'Muon_pfRelIso04_all', 'Muon_dxy', 'Muon_dz', 'Muon_dxyErr', 'Muon_dzErr']
ramas_el = ['run', 'luminosityBlock', 'event', 'nElectron','Electron_pt', 'Electron_eta', 'Electron_phi', 'Electron_charge','Electron_pfRelIso03_all', 'Electron_dxy', 'Electron_dz', 'Electron_dxyErr', 'Electron_dzErr']

# Señal de Higgs (contiene 4 leptones, extraemos tanto muones como electrones)
df_higgs_mu = tree_higgs.arrays(ramas_mu, library="ak")
df_higgs_el = tree_higgs.arrays(ramas_el, library="ak")

# Fondo ZZ → 2e2μ (contiene ambos, extraemos muones y electrones)
df_zz_2e2mu_mu = tree_zz_2e2mu.arrays(ramas_mu, library="ak")
df_zz_2e2mu_el = tree_zz_2e2mu.arrays(ramas_el, library="ak")

print("¡Variables de MC extraídas correctamente ")

¡Variables de MC extraídas correctamente 


In [5]:
# Cargar el archivo Parquet guardado de vuelta a un Awkward Array
df_data_el = ak.from_parquet("data_electrones_B_C_filtrados.parquet")
df_data_mu = ak.from_parquet("data_muones_B_C_filtrados.parquet")


In [24]:
# Dividimos las listas en columnas individuales 

# Datos Reales (Canal Mixto)

mask_datos_mu = (
    (ak.num(df_data_mu["Muon_pt"]) >= 2) &
    (ak.all(df_data_mu["Muon_pfRelIso04_all"] >= 0, axis=1)) &
    (df_data_mu["Muon_pt"][:, 0] > 0) & 
    (df_data_mu["Muon_pt"][:, 1] > 0)
)

mask_datos_el = (
    (ak.num(df_data_el["Electron_pt"]) >= 2) &
    (ak.all(df_data_el["Electron_pfRelIso03_all"] >= 0, axis=1)) &
    (df_data_el["Electron_pt"][:, 0] > 0) & 
    (df_data_el["Electron_pt"][:, 1] > 0)
)


f_data_mu = df_data_mu[mask_datos_mu]
f_data_el = df_data_el[mask_datos_el]

# Crear DataFrame temporal solo con los Muones
df_mu_temp = pd.DataFrame({
    "run": ak.to_numpy(f_data_mu["run"]),
    "luminosityBlock": ak.to_numpy(f_data_mu["luminosityBlock"]),
    "event": ak.to_numpy(f_data_mu["event"]),
    # Muón 1
    "pt_mu1": ak.to_numpy(f_data_mu["Muon_pt"][:, 0]), "eta_mu1": ak.to_numpy(f_data_mu["Muon_eta"][:, 0]),
    "phi_mu1": ak.to_numpy(f_data_mu["Muon_phi"][:, 0]), "charge_mu1": ak.to_numpy(f_data_mu["Muon_charge"][:, 0]),
    "iso_mu1": ak.to_numpy(f_data_mu["Muon_pfRelIso04_all"][:, 0]), "dxy_mu1": ak.to_numpy(f_data_mu["Muon_dxy"][:, 0]),
    "dz_mu1": ak.to_numpy(f_data_mu["Muon_dz"][:, 0]), "dxyErr_mu1": ak.to_numpy(f_data_mu["Muon_dxyErr"][:, 0]), "dzErr_mu1": ak.to_numpy(f_data_mu["Muon_dzErr"][:, 0]),
    # Muón 2
    "pt_mu2": ak.to_numpy(f_data_mu["Muon_pt"][:, 1]), "eta_mu2": ak.to_numpy(f_data_mu["Muon_eta"][:, 1]),
    "phi_mu2": ak.to_numpy(f_data_mu["Muon_phi"][:, 1]), "charge_mu2": ak.to_numpy(f_data_mu["Muon_charge"][:, 1]),
    "iso_mu2": ak.to_numpy(f_data_mu["Muon_pfRelIso04_all"][:, 1]), "dxy_mu2": ak.to_numpy(f_data_mu["Muon_dxy"][:, 1]),
    "dz_mu2": ak.to_numpy(f_data_mu["Muon_dz"][:, 1]), "dxyErr_mu2": ak.to_numpy(f_data_mu["Muon_dxyErr"][:, 1]), "dzErr_mu2": ak.to_numpy(f_data_mu["Muon_dzErr"][:, 1])
})

# Crear DataFrame temporal solo con los Electrones
df_el_temp = pd.DataFrame({
    "run": ak.to_numpy(f_data_el["run"]),
    "luminosityBlock": ak.to_numpy(f_data_el["luminosityBlock"]),
    "event": ak.to_numpy(f_data_el["event"]),
    # Electrón 1
    "pt_el1": ak.to_numpy(f_data_el["Electron_pt"][:, 0]), "eta_el1": ak.to_numpy(f_data_el["Electron_eta"][:, 0]),
    "phi_el1": ak.to_numpy(f_data_el["Electron_phi"][:, 0]), "charge_el1": ak.to_numpy(f_data_el["Electron_charge"][:, 0]),
    "iso_el1": ak.to_numpy(f_data_el["Electron_pfRelIso03_all"][:, 0]), "dxy_el1": ak.to_numpy(f_data_el["Electron_dxy"][:, 0]),
    "dz_el1": ak.to_numpy(f_data_el["Electron_dz"][:, 0]), "dxyErr_el1": ak.to_numpy(f_data_el["Electron_dxyErr"][:, 0]), "dzErr_el1": ak.to_numpy(f_data_el["Electron_dzErr"][:, 0]),
    # Electrón 2
    "pt_el2": ak.to_numpy(f_data_el["Electron_pt"][:, 1]), "eta_el2": ak.to_numpy(f_data_el["Electron_eta"][:, 1]),
    "phi_el2": ak.to_numpy(f_data_el["Electron_phi"][:, 1]), "charge_el2": ak.to_numpy(f_data_el["Electron_charge"][:, 1]),
    "iso_el2": ak.to_numpy(f_data_el["Electron_pfRelIso03_all"][:, 1]), "dxy_el2": ak.to_numpy(f_data_el["Electron_dxy"][:, 1]),
    "dz_el2": ak.to_numpy(f_data_el["Electron_dz"][:, 1]), "dxyErr_el2": ak.to_numpy(f_data_el["Electron_dxyErr"][:, 1]), "dzErr_el2": ak.to_numpy(f_data_el["Electron_dzErr"][:, 1])
})

# Cruzar ambos DataFrames 

df_2mu2el_datos = pd.merge(df_mu_temp, df_el_temp, on=["run", "luminosityBlock", "event"])

print("Eventos totales en canal mixto (Datos Reales):", len(df_2mu2el_datos))

Eventos totales en canal mixto (Datos Reales): 467


In [8]:
# Guardar los DataFrames procesados del canal mixto en disco (formato Parquet)
df_2mu2el_datos.to_parquet("df_2mu2el_datos_filtrados.parquet")
# Limpiar la RAM de los objetos temporales y DataFrames ya guardados
del df_2mu2el_datos
if 'df_mu_temp' in locals(): del df_mu_temp
if 'df_el_temp' in locals(): del df_el_temp

In [18]:
# Señal Higgs (Canal Mixto)

mask_higgs_mu = (
    (ak.num(df_higgs_mu["Muon_pt"]) >= 2) & 
    (ak.all(df_higgs_mu["Muon_pfRelIso04_all"] >= 0, axis=1))
)

mask_higgs_el = (
    (ak.num(df_higgs_el["Electron_pt"]) >= 2) &
    (ak.all(df_higgs_el["Electron_pfRelIso03_all"] >= 0, axis=1))
)

f_higgs_mu = df_higgs_mu[mask_higgs_mu]
f_higgs_el = df_higgs_el[mask_higgs_el]

# Como f_higgs ya solo tiene eventos con >= 2 partículas, 
# podemos agregar con seguridad la condición para eliminar los -999 (o valores <= 0) en los primeros dos leptones.

mask_valores_mu = (
    (f_higgs_mu["Muon_pt"][:, 0] > 0) &  
    (f_higgs_mu["Muon_pt"][:, 1] > 0)
)

mask_valores_el = (
    (f_higgs_el["Electron_pt"][:, 0] > 0) &  
    (f_higgs_el["Electron_pt"][:, 1] > 0)
)

f_higgs_mu = f_higgs_mu[mask_valores_mu]
f_higgs_el = f_higgs_el[mask_valores_el]

# DataFrame temporal de Muones para Higgs

df_higgs_mu_temp = pd.DataFrame({
    "run": ak.to_numpy(f_higgs_mu["run"]),
    "luminosityBlock": ak.to_numpy(f_higgs_mu["luminosityBlock"]),
    "event": ak.to_numpy(f_higgs_mu["event"]),
    # Muón 1
    "pt_mu1": ak.to_numpy(f_higgs_mu["Muon_pt"][:, 0]), "eta_mu1": ak.to_numpy(f_higgs_mu["Muon_eta"][:, 0]),
    "phi_mu1": ak.to_numpy(f_higgs_mu["Muon_phi"][:, 0]), "charge_mu1": ak.to_numpy(f_higgs_mu["Muon_charge"][:, 0]),
    "iso_mu1": ak.to_numpy(f_higgs_mu["Muon_pfRelIso04_all"][:, 0]), "dxy_mu1": ak.to_numpy(f_higgs_mu["Muon_dxy"][:, 0]),
    "dz_mu1": ak.to_numpy(f_higgs_mu["Muon_dz"][:, 0]), "dxyErr_mu1": ak.to_numpy(f_higgs_mu["Muon_dxyErr"][:, 0]), "dzErr_mu1": ak.to_numpy(f_higgs_mu["Muon_dzErr"][:, 0]),
    # Muón 2
    "pt_mu2": ak.to_numpy(f_higgs_mu["Muon_pt"][:, 1]), "eta_mu2": ak.to_numpy(f_higgs_mu["Muon_eta"][:, 1]),
    "phi_mu2": ak.to_numpy(f_higgs_mu["Muon_phi"][:, 1]), "charge_mu2": ak.to_numpy(f_higgs_mu["Muon_charge"][:, 1]),
    "iso_mu2": ak.to_numpy(f_higgs_mu["Muon_pfRelIso04_all"][:, 1]), "dxy_mu2": ak.to_numpy(f_higgs_mu["Muon_dxy"][:, 1]),
    "dz_mu2": ak.to_numpy(f_higgs_mu["Muon_dz"][:, 1]), "dxyErr_mu2": ak.to_numpy(f_higgs_mu["Muon_dxyErr"][:, 1]), "dzErr_mu2": ak.to_numpy(f_higgs_mu["Muon_dzErr"][:, 1])
})

# DataFrame temporal de Electrones para Higgs

df_higgs_el_temp = pd.DataFrame({
    "run": ak.to_numpy(f_higgs_el["run"]),
    "luminosityBlock": ak.to_numpy(f_higgs_el["luminosityBlock"]),
    "event": ak.to_numpy(f_higgs_el["event"]),
    # Electrón 1
    "pt_el1": ak.to_numpy(f_higgs_el["Electron_pt"][:, 0]), "eta_el1": ak.to_numpy(f_higgs_el["Electron_eta"][:, 0]),
    "phi_el1": ak.to_numpy(f_higgs_el["Electron_phi"][:, 0]), "charge_el1": ak.to_numpy(f_higgs_el["Electron_charge"][:, 0]),
    "iso_el1": ak.to_numpy(f_higgs_el["Electron_pfRelIso03_all"][:, 0]), "dxy_el1": ak.to_numpy(f_higgs_el["Electron_dxy"][:, 0]),
    "dz_el1": ak.to_numpy(f_higgs_el["Electron_dz"][:, 0]), "dxyErr_el1": ak.to_numpy(f_higgs_el["Electron_dxyErr"][:, 0]), "dzErr_el1": ak.to_numpy(f_higgs_el["Electron_dzErr"][:, 0]),
    # Electrón 2
    "pt_el2": ak.to_numpy(f_higgs_el["Electron_pt"][:, 1]), "eta_el2": ak.to_numpy(f_higgs_el["Electron_eta"][:, 1]),
    "phi_el2": ak.to_numpy(f_higgs_el["Electron_phi"][:, 1]), "charge_el2": ak.to_numpy(f_higgs_el["Electron_charge"][:, 1]),
    "iso_el2": ak.to_numpy(f_higgs_el["Electron_pfRelIso03_all"][:, 1]), "dxy_el2": ak.to_numpy(f_higgs_el["Electron_dxy"][:, 1]),
    "dz_el2": ak.to_numpy(f_higgs_el["Electron_dz"][:, 1]), "dxyErr_el2": ak.to_numpy(f_higgs_el["Electron_dxyErr"][:, 1]), "dzErr_el2": ak.to_numpy(f_higgs_el["Electron_dzErr"][:, 1])
})

# Cruce final
df_2mu2el_higgs = pd.merge(df_higgs_mu_temp, df_higgs_el_temp, on=["run", "luminosityBlock", "event"])

print("Eventos totales en canal mixto (2mu2el) para Señal de Higgs:", len(df_2mu2el_higgs))


Eventos totales en canal mixto (2mu2el) para Señal de Higgs: 24965


In [19]:
# Guardar los DataFrames procesados del canal mixto en disco (formato Parquet)
df_2mu2el_higgs.to_parquet("df_2mu2el_higgs_filtrados.parquet")
# Limpiar la RAM de los objetos temporales y DataFrames ya guardados
del df_2mu2el_higgs
for var in ['df_higgs_mu_temp', 'df_higgs_el_temp', 'f_higgs_mu', 'f_higgs_el', 'df_mu_temp', 'df_el_temp']:
    if var in locals():
        del locals()[var]
gc.collect()

print("Archivo Parquet guardado y memoria RAM liberada con éxito.")

Archivo Parquet guardado y memoria RAM liberada con éxito.


In [ ]:
# Fondo ZZ → 2e2μ (contiene ambos, extraemos muones y electrones)
df_zz_2e2mu_mu = tree_zz_2e2mu.arrays(ramas_mu, library="ak")
df_zz_2e2mu_el = tree_zz_2e2mu.arrays(ramas_el, library="ak")

In [22]:
# Fondo ZZ → 2e2μ (Canal Mixto)

mask_zz_mu = (
    (ak.num(df_zz_2e2mu_mu["Muon_pt"]) >= 2) & 
    (ak.all(df_zz_2e2mu_mu["Muon_pfRelIso04_all"] >= 0, axis=1))
)
mask_zz_el = (
    (ak.num(df_zz_2e2mu_el["Electron_pt"]) >= 2) &
    (ak.all(df_zz_2e2mu_el["Electron_pfRelIso03_all"] >= 0, axis=1))
)

f_zz_mu = df_zz_2e2mu_mu[mask_zz_mu]
f_zz_el = df_zz_2e2mu_el[mask_zz_el]

mask_valores_zz_mu = (
    (f_zz_mu["Muon_pt"][:, 0] > 0) &  
    (f_zz_mu["Muon_pt"][:, 1] > 0)
)

mask_valores_zz_el = (
    (f_zz_el["Electron_pt"][:, 0] > 0) &  
    (f_zz_el["Electron_pt"][:, 1] > 0)
)

f_zz_mu = f_zz_mu[mask_valores_zz_mu]
f_zz_el = f_zz_el[mask_valores_zz_el]

# DataFrame temporal de Muones para Fondo ZZ

df_zz_mu_temp = pd.DataFrame({
    "run": ak.to_numpy(f_zz_mu["run"]),
    "luminosityBlock": ak.to_numpy(f_zz_mu["luminosityBlock"]),
    "event": ak.to_numpy(f_zz_mu["event"]),
    # Muón 1
    "pt_mu1": ak.to_numpy(f_zz_mu["Muon_pt"][:, 0]), "eta_mu1": ak.to_numpy(f_zz_mu["Muon_eta"][:, 0]),
    "phi_mu1": ak.to_numpy(f_zz_mu["Muon_phi"][:, 0]), "charge_mu1": ak.to_numpy(f_zz_mu["Muon_charge"][:, 0]),
    "iso_mu1": ak.to_numpy(f_zz_mu["Muon_pfRelIso04_all"][:, 0]), "dxy_mu1": ak.to_numpy(f_zz_mu["Muon_dxy"][:, 0]),
    "dz_mu1": ak.to_numpy(f_zz_mu["Muon_dz"][:, 0]), "dxyErr_mu1": ak.to_numpy(f_zz_mu["Muon_dxyErr"][:, 0]), "dzErr_mu1": ak.to_numpy(f_zz_mu["Muon_dzErr"][:, 0]),
    # Muón 2
    "pt_mu2": ak.to_numpy(f_zz_mu["Muon_pt"][:, 1]), "eta_mu2": ak.to_numpy(f_zz_mu["Muon_eta"][:, 1]),
    "phi_mu2": ak.to_numpy(f_zz_mu["Muon_phi"][:, 1]), "charge_mu2": ak.to_numpy(f_zz_mu["Muon_charge"][:, 1]),
    "iso_mu2": ak.to_numpy(f_zz_mu["Muon_pfRelIso04_all"][:, 1]), "dxy_mu2": ak.to_numpy(f_zz_mu["Muon_dxy"][:, 1]),
    "dz_mu2": ak.to_numpy(f_zz_mu["Muon_dz"][:, 1]), "dxyErr_mu2": ak.to_numpy(f_zz_mu["Muon_dxyErr"][:, 1]), "dzErr_mu2": ak.to_numpy(f_zz_mu["Muon_dzErr"][:, 1])
})

# DataFrame temporal de Electrones para Fondo ZZ

df_zz_el_temp = pd.DataFrame({
    "run": ak.to_numpy(f_zz_el["run"]),
    "luminosityBlock": ak.to_numpy(f_zz_el["luminosityBlock"]),
    "event": ak.to_numpy(f_zz_el["event"]),
    # Electrón 1
    "pt_el1": ak.to_numpy(f_zz_el["Electron_pt"][:, 0]), "eta_el1": ak.to_numpy(f_zz_el["Electron_eta"][:, 0]),
    "phi_el1": ak.to_numpy(f_zz_el["Electron_phi"][:, 0]), "charge_el1": ak.to_numpy(f_zz_el["Electron_charge"][:, 0]),
    "iso_el1": ak.to_numpy(f_zz_el["Electron_pfRelIso03_all"][:, 0]), "dxy_el1": ak.to_numpy(f_zz_el["Electron_dxy"][:, 0]),
    "dz_el1": ak.to_numpy(f_zz_el["Electron_dz"][:, 0]), "dxyErr_el1": ak.to_numpy(f_zz_el["Electron_dxyErr"][:, 0]), "dzErr_el1": ak.to_numpy(f_zz_el["Electron_dzErr"][:, 0]),
    # Electrón 2
    "pt_el2": ak.to_numpy(f_zz_el["Electron_pt"][:, 1]), "eta_el2": ak.to_numpy(f_zz_el["Electron_eta"][:, 1]),
    "phi_el2": ak.to_numpy(f_zz_el["Electron_phi"][:, 1]), "charge_el2": ak.to_numpy(f_zz_el["Electron_charge"][:, 1]),
    "iso_el2": ak.to_numpy(f_zz_el["Electron_pfRelIso03_all"][:, 1]), "dxy_el2": ak.to_numpy(f_zz_el["Electron_dxy"][:, 1]),
    "dz_el2": ak.to_numpy(f_zz_el["Electron_dz"][:, 1]), "dxyErr_el2": ak.to_numpy(f_zz_el["Electron_dxyErr"][:, 1]), "dzErr_el2": ak.to_numpy(f_zz_el["Electron_dzErr"][:, 1])
})

# Cruce final

df_2mu2el_zz = pd.merge(df_zz_mu_temp, df_zz_el_temp, on=["run", "luminosityBlock", "event"])

print("Eventos totales en canal mixto (2mu2el) para Fondo ZZ:", len(df_2mu2el_zz))

Eventos totales en canal mixto (2mu2el) para Fondo ZZ: 163295


In [23]:
# 1. Guardar el DataFrame procesado del fondo ZZ en disco (formato Parquet)
df_2mu2el_zz.to_parquet("df_2mu2el_zz_filtrados.parquet")

# 2. Limpiar la RAM de los objetos temporales y DataFrames ya guardados
del df_2mu2el_zz

# Limpieza segura de variables temporales si existen en el entorno
for var in ['df_zz_mu_temp', 'df_zz_el_temp', 'f_zz_mu', 'f_zz_el', 'df_zz_2e2mu_mu', 'df_zz_2e2mu_el']:
    if var in locals():
        del locals()[var]

# Forzar la recolección de basura para liberar la memoria RAM
gc.collect()

print("Archivo Parquet del fondo ZZ guardado y memoria RAM liberada con éxito.")

Archivo Parquet del fondo ZZ guardado y memoria RAM liberada con éxito.
